<a href="https://colab.research.google.com/github/hoanganh1105/scann-approximate-nearest-neighbor/blob/main/Minichatbot_6xScaNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 🛠️ 1. Cài đặt Thư viện (FINAL FIX: ScaNN + JAX + Numpy)
import os
import sys
import subprocess

print("📦 Đang dọn dẹp môi trường lần cuối...")

# 1. Gỡ bỏ sạch sẽ các thư viện đang đánh nhau
# (Gỡ cả jax và ml_dtypes để cài lại bản khớp nhau)
pkgs = ["numpy", "scann", "tensorflow", "ml_dtypes", "jax", "jaxlib", "sentence-transformers"]
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y"] + pkgs)

print("📦 Đang cài đặt lại theo thứ tự nghiêm ngặt...")

commands = [
    # 1. Cài ml_dtypes MỚI trước (Để thỏa mãn JAX)
    "pip install 'ml_dtypes>=0.5.0'",

    # 2. Cài Numpy CŨ (Để thỏa mãn ScaNN)
    "pip install 'numpy<2.0.0'",

    # 3. Cài ScaNN và Tensorflow
    "pip install scann==1.3.2 tensorflow",

    # 4. Cài các thư viện NLP còn lại
    "pip install sentence-transformers wikipedia-api neo4j google-generativeai langchain fastcoref accelerate"
]

for cmd in commands:
    print(f"   -> Running: {cmd}")
    subprocess.check_call(cmd, shell=True)

# Config môi trường
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import warnings
warnings.filterwarnings("ignore")

print("\n✅ ĐÃ SỬA XONG MỌI XUNG ĐỘT.")
print("🛑 BẮT BUỘC: Vào Menu 'Runtime' -> 'Restart Session' (Khởi động lại phiên) NGAY BÂY GIỜ!")

📦 Đang dọn dẹp môi trường lần cuối...
📦 Đang cài đặt lại theo thứ tự nghiêm ngặt...
   -> Running: pip install 'ml_dtypes>=0.5.0'
   -> Running: pip install 'numpy<2.0.0'
   -> Running: pip install scann==1.3.2 tensorflow
   -> Running: pip install sentence-transformers wikipedia-api neo4j google-generativeai langchain fastcoref accelerate

✅ ĐÃ SỬA XONG MỌI XUNG ĐỘT.
🛑 BẮT BUỘC: Vào Menu 'Runtime' -> 'Restart Session' (Khởi động lại phiên) NGAY BÂY GIỜ!


In [ ]:
# @title ⚙️ FIX: Nâng cấp Google Generative AI SDK
print("📦 Đang nâng cấp google-generativeai...")
!pip install -U google-generativeai
print("✅ Nâng cấp hoàn tất!")

📦 Đang nâng cấp google-generativeai...
✅ Nâng cấp hoàn tất!


In [ ]:
# @title 🔗 2. Cấu hình Kết nối
import os
from google.colab import userdata
import google.generativeai as genai
from neo4j import GraphDatabase

try:
    # 1. Lấy Key từ Secrets
    GEMINI_KEY = userdata.get('GEMINI_API_KEY')
    NEO4J_URI = userdata.get('NEO4J_URI')
    NEO4J_USER = userdata.get('NEO4J_USER')
    NEO4J_PASS = userdata.get('NEO4J_PASS')

    # 2. Cấu hình Gemini
    genai.configure(api_key=GEMINI_KEY)
    llm_model = genai.GenerativeModel('models/gemini-2.5-flash')
    print("✅ Gemini API: OK")

    # 3. Cấu hình Neo4j Driver
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
    driver.verify_connectivity()
    print("✅ Neo4j AuraDB: OK (Driver đã sẵn sàng)")

except Exception as e:
    print(f"❌ LỖI KẾT NỐI: {e}")
    print("👉 Hãy kiểm tra lại mục 'Secrets' (biểu tượng chìa khóa) bên trái màn hình.")

✅ Gemini API: OK
✅ Neo4j AuraDB: OK (Driver đã sẵn sàng)


In [ ]:
# @title 🤖 3. Tải Models
from sentence_transformers import CrossEncoder, SentenceTransformer, util
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from fastcoref import FCoref
import torch
import os

# 1. Tự động phát hiện thiết bị
if torch.cuda.is_available():
    device = "cuda"
    print("🚀 Phát hiện GPU: Đang kích hoạt chế độ Tăng tốc.")
else:
    device = "cpu"
    print("🐢 Không có GPU: Đang chạy chế độ CPU (Sẽ chậm hơn ở khâu trích xuất).")

print("⏳ Đang tải models...")

# 2. Embedding Model (ScaNN/Vector Search)
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=device)

# 3. REBEL Model (Extraction)
rebel_tokenizer = AutoTokenizer.from_pretrained("Babelscape/rebel-large")
rebel_model = AutoModelForSeq2SeqLM.from_pretrained("Babelscape/rebel-large").to(device)

# 4. Coreference Model
coref_model = FCoref(device=device)

# 5. Cross-Encoder (Reranking)
rerank_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device)

print(f"✅ TẤT CẢ MODELS ĐÃ SẴN SÀNG TRÊN: {device.upper()}!")

🐢 Không có GPU: Đang chạy chế độ CPU (Sẽ chậm hơn ở khâu trích xuất).
⏳ Đang tải models...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✅ TẤT CẢ MODELS ĐÃ SẴN SÀNG TRÊN: CPU!


In [ ]:
# @title ⚙️ Helper & 1️⃣ GIAI ĐOẠN 1: ScaNN Chunking
import scann
import numpy as np
import re
from tqdm.notebook import tqdm
import wikipediaapi

# --- 0. SCANN HELPER  ---
def build_scann_index(vectors, num_neighbors=10):
    if vectors is None or len(vectors) == 0: return None
    n_data = len(vectors)
    norms = np.linalg.norm(vectors, axis=1)[:, np.newaxis]
    vectors = vectors / (norms + 1e-9)
    builder = scann.scann_ops_pybind.builder(vectors, num_neighbors, "dot_product")
    if n_data < 200:
        return builder.score_brute_force().build()
    else:
        n_leaves = int(np.sqrt(n_data))
        return builder.tree(num_leaves=n_leaves, num_leaves_to_search=max(1, n_leaves//2), training_sample_size=n_data) \
            .score_ah(2, anisotropic_quantization_threshold=0.2).reorder(num_neighbors).build()

# --- 1. CHUNKING LOGIC ---
def scann_semantic_chunking(text, model, threshold=0.5):
    sentences = re.split(r'(?<=[.?!])\s+', text)
    if not sentences: return []
    embeddings = model.encode(sentences, show_progress_bar=False)
    searcher = build_scann_index(embeddings, num_neighbors=5)
    final_chunks = []
    current_group = [sentences[0]]
    current_vec = embeddings[0]

    for i in range(1, len(sentences)):
        sim_score = np.dot(current_vec, embeddings[i]) / (np.linalg.norm(current_vec) * np.linalg.norm(embeddings[i]) + 1e-9)
        if sim_score >= threshold:
            current_group.append(sentences[i])
            current_vec = (current_vec + embeddings[i]) / 2.0
        else:
            final_chunks.append(" ".join(current_group))
            current_group = [sentences[i]]
            current_vec = embeddings[i]

    if current_group: final_chunks.append(" ".join(current_group))
    return final_chunks

# --- MAIN LOOP ---
TOPICS = ["Leonardo da Vinci", "List of works by Leonardo da Vinci", "Mona Lisa", "The Last Supper (Leonardo da Vinci)", "Science and inventions of Leonardo da Vinci"]
wiki = wikipediaapi.Wikipedia(user_agent='GraphRAG/v2', language='en')
all_chunks = []
global_chunk_id = 0

print("🚀 Bắt đầu ScaNN Pipeline Giai đoạn 1...")
for topic in tqdm(TOPICS):
    page = wiki.page(topic)
    if page.exists():
        chunks = scann_semantic_chunking(page.text[:10000], embedding_model) # Lấy 10k ký tự
        for c in chunks:
            if len(c) > 30:
                # Đảm bảo có source_topic
                all_chunks.append({"chunk_id": f"chk_{global_chunk_id}", "source_topic": topic, "content": c, "type": "text"})
                global_chunk_id += 1

print(f"✅ Đã tạo {len(all_chunks)} chunks sử dụng ScaNN Logic.")

🚀 Bắt đầu ScaNN Pipeline Giai đoạn 1...


  0%|          | 0/5 [00:00<?, ?it/s]

✅ Đã tạo 180 chunks sử dụng ScaNN Logic.


In [ ]:
# @title 5. Giai đoạn trích xuất Graph
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch
import sys

if 'rebel_model' not in globals():
    raise NameError("❌ LỖI: Rebel Model chưa được tải. Hãy chạy Cell 3 trước.")

# 1. Hàm parse kết quả
def extract_rebel_triplets(text):
    triplets = []
    text = text.strip()
    text = text.replace("<s>", "").replace("<pad>", "").replace("</s>", "")

    tokens = text.split()
    relation, subject, object_ = '', '', ''
    current = 'x'

    for token in tokens:
        if token == "<triplet>":
            current = 't'
            if relation != '': triplets.append({'subject': subject.strip(), 'relation': relation.strip(), 'object': object_.strip()})
            relation = ''; subject = ''
        elif token == "<subj>":
            current = 's'
            if relation != '': triplets.append({'subject': subject.strip(), 'relation': relation.strip(), 'object': object_.strip()})
            object_ = ''
        elif token == "<obj>":
            current = 'o'; relation = ''
        else:
            if current == 't': subject += ' ' + token
            elif current == 's': object_ += ' ' + token
            elif current == 'o': relation += ' ' + token

    if relation != '' and subject != '' and object_ != '':
        triplets.append({'subject': subject.strip(), 'relation': relation.strip(), 'object': object_.strip()})
    return triplets

gen_kwargs = {
    "max_length": 256,
    "length_penalty": 0,
    "num_beams": 3,
    "num_return_sequences": 1,
}

knowledge_graph_triples = [] # Reset triples
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"🚀 Bắt đầu trích xuất từ {len(all_chunks)} chunks bằng REBEL...")

# 2. Vòng lặp chính
for chunk in tqdm(all_chunks):
    try:
        model_inputs = rebel_tokenizer(chunk['content'], max_length=256, padding=True, truncation=True, return_tensors='pt')
        input_ids = model_inputs['input_ids'].to(device)
        attention_mask = model_inputs['attention_mask'].to(device)

        generated_tokens = rebel_model.generate(
            input_ids,
            attention_mask=attention_mask,
            **gen_kwargs
        )

        decoded_text = rebel_tokenizer.batch_decode(generated_tokens, skip_special_tokens=False)[0]
        extracted = extract_rebel_triplets(decoded_text)

        for t in extracted:
            t['source_chunk_id'] = chunk['chunk_id']
            knowledge_graph_triples.append(t)

    except Exception as e:
        # print(f"Lỗi trích xuất REBEL: {e}")
        pass

print(f"\n✅ Trích xuất được {len(knowledge_graph_triples)} triples.")
print("👉 Bây giờ bạn có thể chạy Cell 6 (ScaNN Cleaning & Prediction) để tiếp tục.")

🚀 Bắt đầu trích xuất từ 180 chunks bằng REBEL...


  0%|          | 0/180 [00:00<?, ?it/s]


✅ Trích xuất được 221 triples.
👉 Bây giờ bạn có thể chạy Cell 6 (ScaNN Cleaning & Prediction) để tiếp tục.


In [ ]:
# @title ⚙️ 6. GIAI ĐOẠN 2-3 & 4: ScaNN Cleaning, Prediction & Indexing
from tqdm.notebook import tqdm
import numpy as np
import re

if 'knowledge_graph_triples' not in globals() or len(knowledge_graph_triples) == 0:
    raise ValueError("❌ LỖI: knowledge_graph_triples rỗng! Hãy kiểm tra Cell 5.")

# --- GĐ 2: CLEANING (Gom nhóm thực thể trùng bằng ScaNN) ---
print("🔹 Giai đoạn 2: Bắt đầu ScaNN Entity Resolution...")

def is_numeric(text): return bool(re.search(r'\d', text))
entities = list(set([t['subject'] for t in knowledge_graph_triples] + [t['object'] for t in knowledge_graph_triples]))
entity_vecs = embedding_model.encode(entities, show_progress_bar=False)
entity_searcher = build_scann_index(entity_vecs, num_neighbors=10)
canonical_map = {}
visited = set()
for i, ent in enumerate(tqdm(entities, desc="Cleaning Entities")):
    if i in visited: continue

    # ScaNN Search: Ngưỡng rất cao cho việc gộp tên (0.96)
    neighbors, dists = entity_searcher.search(entity_vecs[i], final_num_neighbors=10)

    cluster = []
    for idx, dist in zip(neighbors, dists):
        if dist > 0.96:
            cluster.append(idx)
            visited.add(idx)

    if cluster:
        cluster_names = [entities[ix] for ix in cluster]
        canonical = max(cluster_names, key=len)
        for name in cluster_names:
            canonical_map[name] = canonical

clean_triples = []
for t in knowledge_graph_triples:
    s_new = canonical_map.get(t['subject'], t['subject'])
    o_new = canonical_map.get(t['object'], t['object'])
    if s_new != o_new:
        clean_triples.append({"subject": s_new, "relation": t['relation'], "object": o_new, "source_chunk_id": t['source_chunk_id']})
print(f"   -> Graph làm sạch: {len(clean_triples)} triples.")


# --- GĐ 3: LINK PREDICTION (Dự đoán liên kết ẩn bằng ScaNN) ---
print("🔹 Giai đoạn 3: ScaNN Link Prediction...")
new_links = []
existing_pairs = set((t['subject'], t['object']) for t in clean_triples)

for i, vec in enumerate(tqdm(entity_vecs, desc="Predicting Links")):
    neighbors, dists = entity_searcher.search(vec, final_num_neighbors=5)

    for idx, dist in zip(neighbors, dists):
        if i == idx: continue

        # Vùng "Semantic Related" (0.85 < dist < 0.96)
        if 0.85 < dist < 0.96:
            e1, e2 = entities[i], entities[idx]

            if (e1, e2) not in existing_pairs and (e2, e1) not in existing_pairs:
                new_links.append({"subject": e1, "relation": "SCANN_SEMANTIC_LINK", "object": e2, "source_chunk_id": "scann_prediction"})
                existing_pairs.add((e1, e2))

clean_triples.extend(new_links)
knowledge_graph_triples = clean_triples # Cập nhật triples đã clean
print(f"   -> Đã thêm {len(new_links)} liên kết tiềm năng.")


# --- GĐ 4: INDEXING MAIN KNOWLEDGE BASE ---
print("🔨 Giai đoạn 4: Building Final ScaNN Index...")

final_entities = list(set([t['subject'] for t in clean_triples] + [t['object'] for t in clean_triples]))

corpus = [c['content'] for c in all_chunks] + final_entities
index_map = {}

for i, item in enumerate(corpus):
    if i < len(all_chunks): index_map[i] = {"type": "text", "content": item}
    else: index_map[i] = {"type": "entity", "content": item}

all_vectors = embedding_model.encode(corpus, show_progress_bar=True)

searcher = build_scann_index(all_vectors, num_neighbors=20)

print("✅ Indexing hoàn tất! Hệ thống Full ScaNN đã sẵn sàng.")

🔹 Giai đoạn 2: Bắt đầu ScaNN Entity Resolution...


Cleaning Entities:   0%|          | 0/269 [00:00<?, ?it/s]

   -> Graph làm sạch: 197 triples.
🔹 Giai đoạn 3: ScaNN Link Prediction...


Predicting Links:   0%|          | 0/269 [00:00<?, ?it/s]

   -> Đã thêm 0 liên kết tiềm năng.
🔨 Giai đoạn 4: Building Final ScaNN Index...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

✅ Indexing hoàn tất! Hệ thống Full ScaNN đã sẵn sàng.


In [ ]:
# @title 💾 Tùy chọn: Lưu Triples ra file để dùng lại
import json
import csv

# 1. Lưu dạng JSON
output_json = "knowledge_graph.json"
with open(output_json, "w", encoding="utf-8") as f:
    json.dump(knowledge_graph_triples, f, indent=4, ensure_ascii=False)
print(f"✅ Đã lưu {len(knowledge_graph_triples)} triples vào file: {output_json}")

# 2. Lưu dạng CSV
output_csv = "knowledge_graph.csv"
with open(output_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["subject", "relation", "object", "source_chunk_id"]) # Header
    for t in knowledge_graph_triples:
        writer.writerow([t['subject'], t['relation'], t['object'], t.get('source_chunk_id', '')])
print(f"✅ Đã lưu file CSV: {output_csv}")

print("👉 Bạn có thể tải file về máy bằng cách mở tab 'Files' bên trái màn hình.")

✅ Đã lưu 197 triples vào file: knowledge_graph.json
✅ Đã lưu file CSV: knowledge_graph.csv
👉 Bạn có thể tải file về máy bằng cách mở tab 'Files' bên trái màn hình.


In [ ]:
# @title 📂 Nạp lại Triples từ file
import json
import os

if os.path.exists("knowledge_graph.json"):
    with open("knowledge_graph.json", "r", encoding="utf-8") as f:
        knowledge_graph_triples = json.load(f)
    print(f"✅ Đã nạp lại {len(knowledge_graph_triples)} triples từ file cũ.")

else:
    print("⚠️ Không tìm thấy file lưu trữ. Hãy chạy lại quy trình trích xuất (Cell 5).")

✅ Đã nạp lại 197 triples từ file cũ.


In [ ]:
# @title ⚙️ 7. GIAI ĐOẠN 5 & 6: Retrieval & Smart Cache (Tối ưu Tốc độ & Top-10)
from sentence_transformers import util
import numpy as np
import time
from google.api_core.exceptions import DeadlineExceeded

# --- Memory Setup ---
memory_data = {"vectors": [], "responses": [], "queries": []}
memory_searcher = None

def update_memory_scann(query_text, query_vec, response_text):
    global memory_searcher
    memory_data["vectors"].append(query_vec)
    memory_data["responses"].append(response_text)
    memory_data["queries"].append(query_text)

    if len(memory_data["vectors"]) > 0:
        vecs = np.array(memory_data["vectors"])
        memory_searcher = build_scann_index(vecs, num_neighbors=1)

def check_smart_cache(new_query, new_query_vec, verbose=False):
    if not memory_searcher: return None
    idx, dists = memory_searcher.search(new_query_vec, final_num_neighbors=1)
    if dists[0] < 0.85: return None

    best_idx = idx[0]
    cached_query_text = memory_data["queries"][best_idx]
    score = rerank_model.predict([new_query, cached_query_text])

    if verbose: print(f"   🔍 Cache Check: Score {score:.2f}")
    if score > 2.5: return memory_data["responses"][best_idx] # Tăng ngưỡng lên 2.5 cho chắc
    return None

def retrieve_context_scann(query_vec, verbose=False):
    # 1. Tìm kiếm rộng (K=50) để lọc
    idx, dists = searcher.search(query_vec, final_num_neighbors=50)

    unique_contents = []
    seen_content = set()

    # 2. Lọc lấy Top-10 Unique Contents (Text + Entity)
    # Ưu tiên Text Chunk để có nhiều thông tin hơn
    for i in idx:
        item = index_map[i]
        content = item['content'].replace('\n', ' ').strip()

        if content not in seen_content:
            prefix = "[TEXT]" if item['type'] == 'text' else "[FACT]"
            unique_contents.append(f"{prefix} {content}")
            seen_content.add(content)

        if len(unique_contents) >= 10: # <--- CHỈ LẤY TOP 10 (Tối ưu tốc độ)
            break

    if verbose: print(f"   📄 Retrieved {len(unique_contents)} chunks for context.")
    return unique_contents

def chat_with_scann(query, verbose=False):
    t0 = time.time()
    query_vec = embedding_model.encode(query)
    t1 = time.time()

    # 1. Smart Cache Check
    try:
        cached = check_smart_cache(query, query_vec, verbose)
        if cached:
            if verbose: print(f"   ⚡ SMART CACHE HIT! ({time.time()-t0:.2f}s)")
            return f"⚡ (Cache): {cached}"
    except: pass

    # 2. Retrieval (Top 10)
    context_list = retrieve_context_scann(query_vec, verbose)
    if not context_list: return "Sorry, no info found."

    full_context = "\n".join(context_list)
    t2 = time.time()

    if verbose:
        print(f"   ⏱️ Encoding: {t1-t0:.2f}s | Retrieval: {t2-t1:.2f}s")
        print("   --- CONTEXT (Top 10) ---")
        print(full_context[:200] + "...")
        print("   ------------------------")

    # 3. Generation (Tối ưu Prompt cho Tốc độ & Tóm tắt)
    prompt = f"""
    You are an expert AI assistant focused on Leonardo da Vinci.
    Based on the provided Context (Top 10 relevant facts), provide a CONCISE and ACCURATE SUMMARY answer in English.

    - Answer directly and briefly (aim for 1-2 paragraphs).
    - Do not mention "the provided text".
    - If the answer is not in the context, say "Information not available".

    CONTEXT:
    {full_context}

    QUESTION: {query}
    ANSWER:"""

    try:
        # Timeout 30s là đủ cho câu trả lời ngắn
        response = llm_model.generate_content(prompt, request_options={'timeout': 30})
        ans = response.text
        update_memory_scann(query, query_vec, ans)
        t3 = time.time()
        if verbose: print(f"   🚀 Generation: {t3-t2:.2f}s | Total: {t3-t0:.2f}s")
        return ans
    except Exception as e:
        return f"❌ API Error: {e}"

# --- CHECK KÍCH HOẠT MODEL ---
print("="*50)
if 'llm_model' in globals():
    try:
        # Test nhanh 1 request cực ngắn để đánh thức model
        print("🚀 Đang khởi động Gemini (Warm-up)...")
        llm_model.generate_content("Hi", request_options={'timeout': 5})
        print(f"✅ KÍCH HOẠT THÀNH CÔNG MODEL: {getattr(llm_model, 'model_name', 'Gemini 2.0 Flash')}")
        print("✅ Hệ thống ScaNN GraphRAG (Fast Mode) đã sẵn sàng!")
    except Exception as e:
         print(f"⚠️ Cảnh báo: Model có thể chưa sẵn sàng hoặc lỗi mạng. {e}")
else:
    print("⚠️ LỖI: Chưa chạy Cell 2 để load model.")
print("="*50)

🚀 Đang khởi động Gemini (Warm-up)...
✅ KÍCH HOẠT THÀNH CÔNG MODEL: models/gemini-2.5-flash
✅ Hệ thống ScaNN GraphRAG (Fast Mode) đã sẵn sàng!


#Nếu lỗi Gemini hãy bỏ qua cell dưới và tiếp tục cell khác (phần 9 nhưng ofline)


In [ ]:
# @title 🧪 9. Kiểm thử Chuyên sâu (Updated for All-ScaNN Architecture)
import time

# --- 1. KIỂM TRA SỰ SẴN SÀNG ---
if 'chat_with_scann' not in globals():
    raise NameError("❌ LỖI: Hàm 'chat_with_scann' chưa được định nghĩa. Hãy chạy Cell 7 trước.")

# --- 2. RESET BỘ NHỚ (Để test công bằng) ---
# Phải reset đúng cấu trúc có key 'queries' cho Smart Cache
global memory_data, memory_searcher
memory_data = {"vectors": [], "responses": [], "queries": []}
memory_searcher = None
print("⚡ Bộ nhớ đệm đã được xóa sạch (Clean Slate).")

# --- 3. DANH SÁCH CÂU HỎI TEST ---
test_cases = [
    "Who is Leonardo da Vinci?",                  # Test #1: Full RAG
    "Where was Leonardo born?",                     # Test #2: Fact Retrieval
    "When was the Mona Lisa painted?",            # Test #3: Fact Retrieval
    "Did Leonardo invent anything?",              # Test #4: Summarization
    "Tell me about Leonardo's flying machine.",   # Test #5: Specific Detail
    "List some famous paintings by Leonardo.",    # Test #6: Listing
    "Did he ever finish the Adoration of the Magi?",# Test #7: Reasoning (Unfinished work)
    "What is the date of his birth?",             # Test #8: Fact Retrieval
    "Why is the Mona Lisa famous?",               # Test #9: Explanation
    "Who is Leonardo da Vinci?"                   # Test #10: Cache Hit Check
]

print(f"🚀 BẮT ĐẦU CHẠY {len(test_cases)} TEST CASES VỚI SCANN ENGINE\n")
print("="*60)

for i, query in enumerate(test_cases):
    print(f"\n❓ CÂU HỎI #{i+1}: {query}")
    print("-" * 30)

    start_t = time.time()

    # Gọi hàm chat_with_scann mới (thay vì chat_system cũ)
    # verbose=True để xem nó tìm thấy chunk nào
    try:
        response = chat_with_scann(query, verbose=True)
    except Exception as e:
        response = f"❌ Lỗi: {e}"

    print("-" * 30)
    print(f"🤖 Bot: {response}")
    print(f"⏱️ Thời gian: {time.time()-start_t:.2f}s")
    print("="*60)

    time.sleep(1) # Nghỉ nhẹ để tránh rate limit API

print("\n✅ BÀI KIỂM THỬ HOÀN TẤT.")

⚡ Bộ nhớ đệm đã được xóa sạch (Clean Slate).
🚀 BẮT ĐẦU CHẠY 10 TEST CASES VỚI SCANN ENGINE


❓ CÂU HỎI #1: Who is Leonardo da Vinci?
------------------------------
   📄 Retrieved 10 chunks for context.
   ⏱️ Encoding: 0.04s | Retrieval: 0.00s
   --- CONTEXT (Top 10) ---
[FACT] Leonardo di ser Piero da Vinci
[TEXT] Leonardo da Vinci (1452–1519) was an Italian polymath, regarded as the epitome of the "Renaissance Man", displaying skills in numerous diverse areas of stu...
   ------------------------
   🚀 Generation: 3.28s | Total: 3.31s
------------------------------
🤖 Bot: Leonardo da Vinci (1452–1519), properly named Leonardo di ser Piero da Vinci, was an Italian polymath of the High Renaissance. He was born in or near Vinci, Italy, and is regarded as the epitome of the "Renaissance Man" due to his diverse skills across numerous fields.

Primarily known for his iconic paintings like the *Mona Lisa* and *The Last Supper*, Leonardo also excelled as a draughtsman, engineer, scientist, the

In [ ]:
# @title 🧪 9. Kiểm thử CHUYÊN SÂU (Offline Retrieval Test)
import time
import numpy as np

# --- 1. KIỂM TRA SỰ SẴN SÀNG ---
if 'retrieve_context_scann' not in globals():
    raise NameError("❌ LỖI: Hàm 'retrieve_context_scann' chưa được định nghĩa. Hãy chạy Cell 7 trước.")

# --- 2. HÀM TEST OFFLINE ---
def offline_retrieval_test(query):
    """Chỉ chạy Retrieval và Context Expansion, bỏ qua LLM và Cache."""
    # Khởi tạo vector query
    query_vec = embedding_model.encode(query)

    # Retrieval (Gọi trực tiếp Giai đoạn 5)
    context_list = retrieve_context_scann(query_vec, verbose=True)

    if not context_list or not "".join(context_list).strip():
        return "❌ Retrieval Failed: No relevant context found."

    # Format Context thành chuỗi để hiển thị
    full_context = "\n".join(context_list)

    # Trả về Context để người dùng tự đánh giá
    return full_context

# --- 3. DANH SÁCH CÂU HỎI TEST ---
test_cases = [
    "Who is Leonardo da Vinci?",                  # Test #1
    "Where was Leonardo born?",                     # Test #2
    "When was the Mona Lisa painted?",            # Test #3
    "Did Leonardo invent anything?",              # Test #4
    "Tell me about Leonardo's flying machine.",   # Test #5
    "List some famous paintings by Leonardo.",    # Test #6
    "Did he ever finish the Adoration of the Magi?",# Test #7
    "What is the date of his birth?",             # Test #8
    "Why is the Mona Lisa famous?",               # Test #9
    "What is Leonardo's full name?"               # Test #10: Ngữ nghĩa gần
]

print(f"🚀 BẮT ĐẦU CHẠY {len(test_cases)} TEST CASES (OFFLINE RETRIEVAL)\n")
print("="*60)

for i, query in enumerate(test_cases):
    print(f"\n❓ CÂU HỎI #{i+1}: {query}")
    print("-" * 30)

    start_t = time.time()

    # Gọi hàm test OFFLINE
    response = offline_retrieval_test(query)

    print("-" * 30)
    print("📦 CONTEXT (Tìm thấy):")
    # In toàn bộ Context tìm thấy
    print(response)
    print(f"\n⏱️ Thời gian: {time.time()-start_t:.2f}s (Chỉ tính Retrieval)")
    print("="*60)

    time.sleep(0.5)

print("\n✅ BÀI KIỂM THỬ HOÀN TẤT.")

🚀 BẮT ĐẦU CHẠY 10 TEST CASES (OFFLINE RETRIEVAL)


❓ CÂU HỎI #1: Who is Leonardo da Vinci?
------------------------------
   📄 Retrieved 10 chunks for context.
------------------------------
📦 CONTEXT (Tìm thấy):
[FACT] Leonardo di ser Piero da Vinci
[TEXT] Leonardo da Vinci (1452–1519) was an Italian polymath, regarded as the epitome of the "Renaissance Man", displaying skills in numerous diverse areas of study. While most famous for his paintings such as the Mona Lisa and the Last Supper, Leonardo is also renowned in the fields of civil engineering, chemistry, geology, geometry, hydrodynamics, mathematics, mechanical engineering, optics, physics, pyrotechnics, and zoology.
[TEXT] Biography Early life (1452–1472) Birth and background Leonardo da Vinci, properly named Leonardo di ser Piero da Vinci ("Leonardo, son of ser Piero from Vinci"), was born on 15 April 1452 in, or close to, the Tuscan hill town of Vinci, Italy 20 miles from Florence. He was born out of wedlock to Piero da Vinc

In [ ]:
# @title 💬 10. Giao diện Chat 1:1 (Kết nối ScaNN Engine - Final)
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import time

# --- 1. KIỂM TRA ENGINE ---
if 'chat_with_scann' not in globals():
    print("❌ Lỗi: Hàm 'chat_with_scann' chưa được định nghĩa.")
    print("👉 Hãy chạy lại Cell 7 (Logic Chatbot) trước nhé!")
else:
    print("✅ Đã kết nối với ScaNN Engine!")

# --- 2. THIẾT KẾ GIAO DIỆN ---
# Header
header = widgets.HTML("""
<div style="background-color:#f0f2f6; padding:15px; border-radius:10px; margin-bottom:10px; border: 1px solid #e0e0e0;">
    <h3 style="color:#2c3e50; margin:0;">🤖 ScaNN GraphRAG Chatbot</h3>
    <p style="color:#7f8c8d; margin:5px 0 0 0; font-size:14px;">Expert on Leonardo da Vinci (Powered by Gemini & ScaNN)</p>
</div>
""")

# Khu vực hiển thị chat
chat_history = widgets.Output(layout={'border': '1px solid #ddd', 'height': '500px', 'overflow_y': 'scroll', 'padding': '15px', 'border_radius': '10px'})

# Ô nhập liệu
user_input = widgets.Text(
    placeholder="Hỏi gì đó về Da Vinci... (VD: Who is he?)",
    layout=widgets.Layout(width='70%')
)

# Nút Gửi
send_btn = widgets.Button(
    description="Gửi ➤",
    button_style='primary',
    layout=widgets.Layout(width='15%')
)

# Nút Reset (Xóa bộ nhớ)
reset_btn = widgets.Button(
    description="🔄 Reset",
    button_style='warning',
    layout=widgets.Layout(width='10%'),
    tooltip="Xóa bộ nhớ đệm và làm mới cuộc trò chuyện"
)

# --- 3. LOGIC XỬ LÝ ---
def on_send_click(_):
    query = user_input.value.strip()
    if not query: return

    user_input.value = "" # Xóa ô nhập

    with chat_history:
        # In câu hỏi người dùng (Căn phải)
        display(HTML(f"""
        <div style='display: flex; justify-content: flex-end; margin-bottom: 10px;'>
            <div style='background-color: #e3f2fd; padding: 10px 15px; border-radius: 15px 15px 0 15px; max-width: 70%; box-shadow: 1px 1px 3px rgba(0,0,0,0.1);'>
                <b>👤 Bạn:</b> {query}
            </div>
        </div>
        """))

        # Hiển thị trạng thái đang nghĩ (tạm thời)
        loading_msg = display(HTML("<div style='color:#999; font-style:italic; margin-left:10px;'>⏳ Bot đang suy nghĩ...</div>"), display_id=True)

        # --- GỌI PIPELINE ---
        start_time = time.time()
        try:
            # Gọi hàm chính: chat_with_scann
            response = chat_with_scann(query, verbose=False)
        except Exception as e:
            response = f"❌ Lỗi hệ thống: {str(e)}"
        end_time = time.time()

        # Xóa dòng loading
        loading_msg.update(HTML(""))

        # In câu trả lời Bot (Căn trái)
        display(HTML(f"""
        <div style='display: flex; justify-content: flex-start; margin-bottom: 20px;'>
            <div style='background-color: #f8f9fa; padding: 15px; border-radius: 15px 15px 15px 0; max-width: 80%; border: 1px solid #eee; box-shadow: 1px 1px 3px rgba(0,0,0,0.05);'>
                <b>🤖 Bot ({end_time - start_time:.2f}s):</b><br>
                <div style='margin-top: 5px; line-height: 1.5;'>{response.replace(chr(10), '<br>')}</div>
            </div>
        </div>
        <script>
            // Tự động cuộn xuống cuối
            var chat_div = this.parentElement;
            chat_div.scrollTop = chat_div.scrollHeight;
        </script>
        """))
        # Hack nhỏ để auto-scroll trong Colab output
        display(Javascript("document.querySelector('.widget-output').scrollTop = document.querySelector('.widget-output').scrollHeight;"))

def on_reset_click(_):
    # Reset biến toàn cục memory
    global memory_data, memory_searcher
    memory_data = {"vectors": [], "responses": [], "queries": []}
    memory_searcher = None

    chat_history.clear_output()
    with chat_history:
        display(HTML("<div style='text-align:center; color:#888; margin: 20px;'><i>🧹 Đã xóa ký ức. Bắt đầu phiên làm việc mới.</i></div>"))

# --- 4. KÍCH HOẠT SỰ KIỆN ---
send_btn.on_click(on_send_click)
user_input.on_submit(on_send_click)
reset_btn.on_click(on_reset_click)

# --- 5. HIỂN THỊ ---
display(header)
display(chat_history)
display(widgets.HBox([user_input, send_btn, reset_btn]))

In [ ]:
# @title 💬 12. Giao diện DỰ PHÒNG Tương tác (Offline Retrieval UI - FIX TYPE ERROR)
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, Javascript
import time

# --- KIỂM TRA SỰ SẴN SÀNG (BẮT BUỘC) ---
if 'retrieve_context_scann' not in globals():
    print("❌ LỖI: Hàm 'retrieve_context_scann' chưa được nạp. Vui lòng chạy Cell 7 trước.")
else:
    print("✅ Giao diện Dự phòng đã sẵn sàng!")

# --- LOGIC DỰ PHÒNG ---
def fallback_query(query):
    """
    Chạy Giai đoạn 5 (ScaNN Retrieval) và hiển thị trực tiếp Context.
    """
    if 'embedding_model' not in globals():
        return "❌ LỖI: Embedding Model chưa được nạp. Vui lòng chạy Cell 3 trước."

    start_time = time.time()

    # 1. ENCODE QUERY (BƯỚC THIẾU)
    query_vec = embedding_model.encode(query)

    # 2. Gọi hàm Retrieval với VECTOR
    context_list = retrieve_context_scann(query_vec, verbose=False)

    end_time = time.time()

    if not context_list or not "".join(context_list).strip():
        return "Sorry, no relevant context found in the ScaNN Index."

    # Format Output cho người dùng dễ đọc
    output = []
    output.append("💡 PHƯƠNG ÁN DỰ PHÒNG (LLM OFFLINE)")
    output.append("-" * 35)
    output.append(f"⏱️ Retrieval Time: {end_time - start_time:.4f}s")
    output.append(f"🔎 CÂU HỎI: {query}")
    output.append("-" * 35)
    output.append("📦 CONTEXT ỨNG VIÊN (TOP CHUNKS TỪ SCANN):")

    for i, item in enumerate(context_list):
        output.append(f"  {i+1}. {item}")

    return "\n".join(output)


# --- UI DỰ PHÒNG ---
fallback_input = widgets.Text(placeholder="Nhập câu hỏi để tìm kiếm trực tiếp...", layout=widgets.Layout(flex='1 0 auto'))
fallback_btn = widgets.Button(description="Tìm kiếm", button_style='info', layout=widgets.Layout(width='15%'))
fallback_output = widgets.Output()

def on_fallback_click(_):
    query = fallback_input.value.strip()
    if not query: return

    with fallback_output:
        clear_output(wait=True)
        print("⏳ Đang chạy ScaNN Retrieval...")

        # Gọi hàm dự phòng
        result = fallback_query(query)

        # Hiển thị kết quả
        clear_output(wait=True)
        display(HTML(f"""
        <pre style='background-color: #f8f8f8; padding: 15px; border-radius: 8px; white-space: pre-wrap; word-wrap: break-word;'>
            {result}
        </pre>
        """))

# --- KÍCH HOẠT ---
fallback_btn.on_click(on_fallback_click)
fallback_input.on_submit(on_fallback_click)

display(HTML("<h2>🚧 FALLBACK MODE: TRUY VẤN TRỰC TIẾP SCANN</h2>"))
display(widgets.HBox([fallback_input, fallback_btn]))
display(fallback_output)